# Section 4: LEC-VEC Trajectory Preparation

## Purpose
Prepare vascular and lymphatic endothelial subsets and export inputs for downstream pseudotime modeling.


## Workflow Overview


In [ ]:
%load_ext autoreload
%autoreload 2

## Setup


In [ ]:
# System utilities
import os
import pickle
from pathlib import Path
from datetime import datetime
import warnings
import time
import math

# Data handling and numerical computation
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn3
import matplotlib.colors as mcolors
from upsetplot import UpSet, from_memberships
from plot_utils import proportion
from utils import plot_upset
from utils import scatter_df
import matplotlib.ticker as ticker

# Single-cell analysis and related packages
import anndata as ad
import scanpy as sc
import squidpy as sq

# Suppress warnings
warnings.filterwarnings('ignore', category=FutureWarning, module='numpy')
warnings.filterwarnings('ignore', category=FutureWarning, module='scanpy')
warnings.filterwarnings('ignore', category=UserWarning, module='scanpy')
warnings.filterwarnings('ignore', category=UserWarning, module='numpy')
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Performance
from joblib import Parallel, delayed

# Function to print the current time with a message
def print_with_time(message):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}")

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

plt.rcParams['pdf.fonttype'] = 42  # Ensures text is stored as text, not paths
plt.rcParams['ps.fonttype'] = 42


In [ ]:
# Imports: load trajectory, spatial, and enrichment-analysis dependencies
import gseapy as gp

In [ ]:
# Imports: load trajectory, spatial, and enrichment-analysis dependencies
import scanpy.external as sce
import phate

In [ ]:
# Imports: load trajectory, spatial, and enrichment-analysis dependencies
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.geometry import Polygon
from shapely.ops import unary_union

def convert2gpd(df):
    # Step 1: Group the DataFrame by 'cell_id' or similar, assuming each cell has a unique ID
    # Ensure that your DataFrame has an identifier for each cell
    grouped = df.groupby('cell_id')

    # Step 2: Create polygons for each group of cell boundaries
    polygons = []

    for cell_id, group in grouped:
        # Extract the x and y coordinates for this cell
        points = group[['vertex_x', 'vertex_y']].values
        
        # Create a Polygon from these points
        # Ensure the points form a valid polygon (e.g., no crossing lines)
        if len(points) > 2:  # A polygon needs at least 3 points
            poly = Polygon(points)
            polygons.append({'cell_id': cell_id, 'geometry': poly})

    # Step 3: Create a GeoPandas DataFrame from the list of polygons
    gdf = gpd.GeoDataFrame(polygons)
    return gdf

## Output Configuration


In [ ]:

# Helper plotting utility: render and export custom color palettes used across figures
def plot_color_palette(color_dict, title, pdf):
    labels = list(color_dict.keys())
    colors = list(color_dict.values())
    num_colors = len(colors)

    # Create a figure and a set of subplots
    fig, ax = plt.subplots(figsize=(5, num_colors // 2))

    # Plot each color as a horizontal bar
    for i, (label, color) in enumerate(zip(labels, colors)):
        ax.barh(i, 1, color=color)
        ax.text(0.5, i, label, va='center', ha='center', fontsize=10, color='white', fontweight='bold')

    # Remove axes
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])

    # Set title
    ax.set_title(title, fontweight='bold')

    # Save the current figure to the PDF
    if pdf:
        pdf.savefig(fig)
    plt.show()
    plt.close(fig)

## Data Loading and Subsetting


In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828',
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

marker_list_df_all = pd.read_csv('../data/marker_list_dev_standardized_short.csv')
marker_list_df = marker_list_df_all.copy()
marker_list_df = marker_list_df.query(f"Annotation not in ['Vascular Endothelial Cells', 'Lymphatic Endothelial Cells']")

# Define cell types and clusters
cell_types = list(marker_list_df.keys())

marker_list = marker_list_df
marker_list = marker_list.groupby('grouped_cts')['Gene'].unique().reset_index()
marker_list = marker_list.set_index('grouped_cts')['Gene'].apply(list).to_dict()

complete_cell_types = list(marker_list_df['grouped_cts'].unique())
marker_genes_dict = marker_list

In [ ]:
# Process samples
samples = ['KS_TMA_1_0026870', 'KS_TMA_2_0026882', 'KS_TMA_3_0027198', 'KS_TMA_4_0026764', 'KS_TMA_5_0026776',
           'KS_TMA_6_0027092', 'KS_TMA_7_0027079', 'KS_TMA_8_0027273', 'KS_TMA_9_0026831', 'KS_TMA_10_0026828', 
           'KS_TMA_11_0026930', 'KS_TMA_12_0026888', 'KS_TMA_13_0027077', 'KS_TMA_14_0027019', 'KS_TMA_15_0033811',
          'KS_TMA_16_0033809']

adata = sc.read_h5ad('../data/KS_adata_preprocessed.h5ad')

adata.obsm["spatial"] = adata.obs[["local_x", "local_y"]].copy().to_numpy()

In [ ]:
# Define viral marker sets and key clustering parameters used in this trajectory subset
KS_lytic_genes = ['KSHV.ORF50', 'KSHV.ORF57', 'KSHV.ORF59', 'KSHV.K9', 'KSHV.ORF65'] 
KS_latent_genes = ['KSHV.ORF71','KSHV.ORF72','KSHV.ORF73',] 
KS_K2_gene = ['KSHV.K2']

nctc_neighbors = 30
n_clusters = 10

In [ ]:
# Define consistent color mappings for subtypes, broad cell types, and niche annotations
sub_cell_types_color_mapping = {
    'Keratinocytes': '#181c82',
    'Differentiated Keratinocytes': '#471fc7',
    'Spinous to Granular Cells': '#034cff',
    'Pilosebaceous Cells': '#bbbde2',
    'Melanocytes': '#00bbbf',

    'Vascular Endothelial Cells': '#a4e000',
    'Lymphatic Endothelial Cells': '#ffb695',
    'Proliferating Lymphatic Endothelial Cells': '#906855',
    'Pericytes': '#9f7704',
    
    'Fibroblasts': '#c7d0c0',
    'Pro-inflammatory Fibroblasts': '#7cd28e',
    'Mesenchymal Fibroblasts': '#3d8e27',
    'Myofibroblasts': '#007c1d',
    'Secretory-papillary Fibroblasts': '#076018',
    'Secretory-reticular Fibroblasts': '#324708',
    
    'Macrophages': '#ff40ff',
    'Dendritic cells': '#ff9300',
    'B-cells': '#f12d00',
    'T-cells': '#941100',
    'Cd4': '#600c09',
    'Cd4 Rgcc': '#3c0c09',
    'Cd8 Exhausted': '#240c09',
}


with PdfPages('../figures/sub_cell_types_color_mapping.pdf') as pdf:
    plot_color_palette(sub_cell_types_color_mapping, "Cell Subtypes", pdf)

In [ ]:
# Define consistent color mappings for subtypes, broad cell types, and niche annotations
broad_cell_types_color_mapping = {\
    'Lymphatic Endothelial Cells': '#ffb695',
    'Macrophages': '#ff40ff',
    'Vascular Endothelial Cells': '#a4e000',
    'Pericytes': '#9f7704',
    'Fibroblasts': '#c7d0c0',
    'T-cells': '#941100',
    'Keratinocytes': '#181c82',
    'Dendritic cells': '#ff9300',
    'Spinous to Granular Cells': '#034cff',
    'Pilosebaceous Cells': '#bbbde2',
    'B-cells': '#f12d00',
    'Melanocytes': '#00bbbf'
}

with PdfPages('../figures/broad_cell_types_color_mapping.pdf') as pdf:
    plot_color_palette(broad_cell_types_color_mapping, "Broad Cell Types", pdf)

In [ ]:
# Define consistent color mappings for subtypes, broad cell types, and niche annotations
niche_colors = {
    "Basal epidermis": "#b299e3",
    "Differentiated epidermis": "#ffd000",
    "Stroma": "#646500",
    "TA VEC stroma": "#00e50c",
    "VEC stroma": "#cccc33",
    "Macrophage stroma": "#00dbf4",
    "T cell stroma": "#0051f9",
    "Immune": "#c100f9",
    "Tumor core": "#450000",
    "Tumor": "#eb0000",
    "Tumor boundary": "#faa0aa"
}


with PdfPages('../figures/niches_color_mapping.pdf') as pdf:
    plot_color_palette(niche_colors, "Niches", pdf)

In [ ]:
# Define stage color palette used in proportion plots and trajectory figures
stage_colors = {
        'nodular': '#ed322f', 'plaque': '#ffdc5e',
        'patch': '#51f512', 'control': 'blue'
    }


with PdfPages('../figures/stage_color_mapping.pdf') as pdf:
    plot_color_palette(stage_colors, "Stages", pdf)

In [ ]:
# Prepare tabular exports required by downstream trajectory scripts
import pandas as pd
import numpy as np

# Step 1: Extract relevant data from the AnnData object
# Assuming adata.obs contains 'path_block_core', 'broad_cell_types', and 'Stage'
df = adata.obs[['path_block_core', 'broad_cell_types', 'Stage']].copy()

# Step 2: Count occurrences of each cell type per sample and stage
cell_counts = df.groupby(['Stage', 'path_block_core', 'broad_cell_types']).size().unstack(fill_value=0)

# Step 3: Calculate proportions
cell_counts['Lymphatic'] = cell_counts.get('Lymphatic Endothelial Cells', 0)
cell_counts['Vascular'] = cell_counts.get('Vascular Endothelial Cells', 0)

# Avoid division by zero
cell_counts['Proportion'] = np.where(cell_counts['Vascular'] > 0,
                                     cell_counts['Lymphatic'] / cell_counts['Vascular'],
                                     0)

# Step 4: Group by Stage and list samples with their proportions
proportions_by_stage = cell_counts.groupby('Stage')['Proportion'].agg(list).reset_index()

In [ ]:
# Prepare tabular exports required by downstream trajectory scripts
import pandas as pd
import numpy as np

# Step 1: Extract relevant data from the AnnData object
df = adata.obs[['path_block_core', 'broad_cell_types', 'Stage']].copy()

# Step 2: Count occurrences of each cell type per sample and stage
cell_counts = df.groupby(['Stage', 'path_block_core', 'broad_cell_types']).size().unstack(fill_value=0)

# Step 3: Calculate proportions
cell_counts['Lymphatic'] = cell_counts.get('Lymphatic Endothelial Cells', 0)
cell_counts['Vascular'] = cell_counts.get('Vascular Endothelial Cells', 0)

# Avoid division by zero and calculate proportions
cell_counts['Proportion'] = np.where(cell_counts['Vascular'] > 0,
                                     cell_counts['Lymphatic'] / cell_counts['Vascular'],
                                     0)

# Step 4: Create a DataFrame with 'Stage', 'path_block_core', and 'Proportion' columns
proportions_df = cell_counts[['Proportion']].reset_index()

# Step 5: Drop rows where Proportion is zero
proportions_df = proportions_df[proportions_df['Proportion'] != 0]

# Step 6: Set 'path_block_core' as the index
proportions_df.set_index('path_block_core', inplace=True)

# Check the range of Proportion values
min_proportion = proportions_df['Proportion'].min()
max_proportion = proportions_df['Proportion'].max()
print(f"Minimum Proportion: {min_proportion}, Maximum Proportion: {max_proportion}")

# Step 7: Create bins for the Proportion column
# Define the bins based on the range of Proportion
bins = [0, 0.1, 1, 5, 10, 20, 30, 40]  # Adjust these values based on your analysis needs
labels = range(len(bins) - 1)  # Bin IDs (0, 1, 2, ...)

# Create a new column 'Bin_ID' based on the 'Proportion'
proportions_df['Bin_ID'] = pd.cut(proportions_df['Proportion'], bins=bins, labels=labels, right=False)

# Optional: Display the resulting DataFrame and check for NaN values
print(proportions_df)
print(proportions_df[proportions_df['Bin_ID'].isna()])

In [ ]:
# Inspect bin-level sample counts in the stage proportion summary table
proportions_df.Bin_ID.value_counts()

In [ ]:
# Summarize bin counts by stage to validate sampling balance
proportions_df.groupby(['Bin_ID', 'Stage']).size()

In [ ]:
# Set neighborhood parameter and construct stage-specific core lists
n=8

# Step 1: Filter the DataFrame for Bin_ID == 2
filtered_df = proportions_df[proportions_df['Bin_ID'].isin([0, 1, 2])]

# Step 2: Group by 'Stage' and sample 8 samples from each group
sampled_df = filtered_df.groupby('Stage').apply(lambda x: x.sample(n=n, random_state=1111) if len(x) >= n else x)

# Step 3: Reset the index and extract 'path_block_core' names
# Assuming 'path_block_core' is part of the index
path_block_core_names = sampled_df.index.tolist()  # This gets the index if it's the name you're looking for

# Convert to dictionary
result_dict = {}
for key, value in path_block_core_names:
    if key not in result_dict:
        result_dict[key] = []  # Initialize a list for new keys
    result_dict[key].append(value)  # Append the value to the list
result_dict

In [ ]:
# Combine selected cores across disease stages for downstream trajectory analysis
keep_cores = result_dict['patch'] + result_dict['plaque'] + result_dict['nodular']

In [ ]:
# Subset AnnData to selected cores/stages prior to lineage-focused trajectory modeling
adata_subset = adata[(adata.obs.path_block_core.isin(keep_cores)) | (adata.obs.Stage == 'control')]
adata_subset.shape

In [ ]:
# QC check: verify stage distribution in the trajectory subset
adata_subset.obs.Stage.value_counts()

In [ ]:
# QC check: inspect broad cell-type composition in the subset
adata_subset.obs.broad_cell_types.value_counts()

In [ ]:
# List available broad cell-type labels prior to lineage filtering
adata_subset.obs.broad_cell_types.unique().tolist()

In [ ]:
# keep_cell_types = ['Lymphatic Endothelial Cells', 'Vascular Endothelial Cells', 'Fibroblasts', 'Macrophages', 'Dendritic cells', 'T-cells']

keep_cell_types = ['Lymphatic Endothelial Cells', 'Vascular Endothelial Cells']

In [ ]:
# Apply lineage filter to keep only vascular and lymphatic endothelial populations
adata_subset = adata_subset[adata_subset.obs.broad_cell_types.isin(keep_cell_types)]

In [ ]:
# Remove viral genes from features before PHATE trajectory embedding
mask = ~adata_subset.var_names.str.startswith("KSHV")
adata_subset = adata_subset[:, mask]


In [ ]:
# Create control-only reference subset for comparative trajectory context
adata_control = adata[adata.obs.Stage.isin(['control'])]

In [ ]:
# QC check: inspect broad cell-type composition in the subset
pd.DataFrame(adata_control.obs.broad_cell_types.value_counts())

In [ ]:
# QC check: inspect broad cell-type composition in the subset
adata_control.obs.broad_cell_types.value_counts().plot(kind='bar', title='Cell Types in Control Samples')

In [ ]:
# QC check: verify stage distribution in the trajectory subset
adata_subset.obs.Stage.value_counts()

In [ ]:
# Inspect subset shape after stage and lineage filtering steps
adata_subset.shape

In [ ]:
# QC check: inspect broad cell-type composition in the subset
adata_subset.obs.broad_cell_types.value_counts()

In [ ]:
# Define consistent color mappings for subtypes, broad cell types, and niche annotations
proportion(adata_subset, group_key='Stage', label_key='niches', palette=niche_colors)

In [ ]:
# Define consistent color mappings for subtypes, broad cell types, and niche annotations
proportion(adata_subset, group_key='Stage', label_key='cell_type', palette=sub_cell_types_color_mapping)

In [ ]:
# Plot stage-by-lineage counts to verify representation before PHATE analysis
adata_subset.obs.groupby(['Stage', 'broad_cell_types']).size().plot(kind='bar')

In [ ]:
%%time
sce.tl.phate(adata_subset, 
            k=20,
            a=None,
            n_jobs = -1,
            #n_pca=50,
            random_state=1111
            )

In [ ]:
# Plot PHATE embedding colored by key annotations for trajectory interpretation
sce.pl.phate(
    adata_subset,
    color='broad_cell_types',
    # show=False
)

In [ ]:
%%time

sc.pp.neighbors(adata_subset, n_neighbors=20, random_state=1111)

In [ ]:
%%time

sc.tl.leiden(
    adata_subset, 
    resolution=0.4,
    random_state=1111,
    
)

In [ ]:
# Inspect Leiden assignments before downstream composition and marker analysis
adata_subset.obs.leiden.nunique()

In [ ]:
# Plot PHATE embedding colored by key annotations for trajectory interpretation
sce.pl.phate(
    adata_subset,
    color='leiden',
    # show=False
)

In [ ]:
# Inspect Leiden assignments before downstream composition and marker analysis
adata_subset.obs.leiden.value_counts()

In [ ]:
# Define consistent color mappings for subtypes, broad cell types, and niche annotations
proportion(adata_subset, group_key='leiden', label_key='niches', palette=niche_colors)

## Visualization and Marker Review


In [ ]:
# Group by 'leiden', 'infection_status', and 'broad_cell_types' and count occurrences
count_data = adata_subset.obs.groupby(['leiden', 'infection_status', 'Stage']).size()

# Create a bar plot
ax = count_data.plot(kind='barh', figsize=(4, 8), color=[stage_colors[idx[2]] for idx in count_data.index])

# Add counts on top of each bar
for p in ax.patches:
    # Extract the corresponding infection status and broad_cell_type for the current bar
    idx = count_data.index[ax.patches.index(p)]
    infection_status = idx[1]  # Get the infection status from the index
    broad_cell_type = idx[2]  # Get the broad cell type from the index
    
    # Set text color based on infection status
    text_color = 'red' if infection_status == 'infected' else 'black'
    
    # Annotate with the determined text color
    ax.annotate(f'{int(p.get_width())}',  # Use get_width() for horizontal bars
                (p.get_width(), p.get_y() + p.get_height() / 2.),  # Correct the annotation position
                ha='left', va='center', color=text_color)  # Set the text color

# Display the plot
plt.ylabel('Count')
plt.title('Counts of Cells by Cluster ID and Stage')
plt.show()

In [ ]:
# Group by 'leiden', 'infection_status', and 'broad_cell_types' and count occurrences
count_data = adata_subset.obs.groupby(['leiden', 'infection_status', 'broad_cell_types']).size()

# Create a bar plot
ax = count_data.plot(kind='barh', figsize=(5, 4), color=[broad_cell_types_color_mapping[idx[2]] for idx in count_data.index])

# Add counts on top of each bar
for p in ax.patches:
    # Extract the corresponding infection status and broad_cell_type for the current bar
    idx = count_data.index[ax.patches.index(p)]
    infection_status = idx[1]  # Get the infection status from the index
    broad_cell_type = idx[2]  # Get the broad cell type from the index
    
    # Set text color based on infection status
    text_color = 'red' if infection_status == 'infected' else 'black'
    
    # Annotate with the determined text color
    ax.annotate(f'{int(p.get_width())}',  # Use get_width() for horizontal bars
                (p.get_width(), p.get_y() + p.get_height() / 2.),  # Correct the annotation position
                ha='left', va='center', color=text_color)  # Set the text color

# Display the plot
plt.ylabel('Count')
plt.title('Counts of Cells by Cluster ID and Cell Type')
plt.show()

In [ ]:
# Plot differential-expression summaries for selected clusters (custom figure configuration)
import matplotlib.pyplot as plt

# Group and count data
count_data = adata_subset.obs.groupby(['leiden', 'infection_status', 'broad_cell_types']).size()

# Get unique leiden cluster IDs
leiden_clusters = sorted(count_data.index.get_level_values('leiden').unique())

# Determine the layout for subplots
n_clusters = len(leiden_clusters)
fig, axes = plt.subplots(n_clusters, 2, figsize=(6, 1.5*n_clusters), sharex=True, sharey=True)

# If there's only one cluster, make sure axes is 2D
if n_clusters == 1:
    axes = axes.reshape(1, 2)

for i, leiden in enumerate(leiden_clusters):
    for j, status in enumerate(['uninfected', 'infected']):
        # Extract data for this leiden and infection status
        data = count_data.loc[leiden]
        if status in data:
            bar_data = data.loc[status]
        else:
            bar_data = pd.Series(dtype=int)  # Empty if status not present

        ax = axes[i, j]
        colors = [broad_cell_types_color_mapping[ct] for ct in bar_data.index]
        bars = bar_data.plot(kind='barh', ax=ax, color=colors)

        ax.set_title(f'Leiden {leiden} - {status.capitalize()}')
        ax.set_xlabel('Count')
        ax.set_ylabel('')

        # Annotate bars
        for p in bars.patches:
            ct = bar_data.index[bars.patches.index(p)]
            text_color = 'red' if status == 'infected' else 'black'
            bars.annotate(f'{int(p.get_width())}',
                          (p.get_width(), p.get_y() + p.get_height() / 2.),
                          ha='left', va='center', color=text_color)

plt.tight_layout()
plt.show()


In [ ]:
# Plot GSEA enrichment overview for selected marker sets
import matplotlib.pyplot as plt

# Group by 'leiden' and 'infection_status' and count occurrences
count_data = adata_subset.obs.groupby(['leiden', 'Stage']).size().unstack(fill_value=0)

# Unique leiden clusters
leiden_clusters = count_data.index

# Create subplots - one for each unique leiden cluster
n_clusters = len(leiden_clusters)
fig, axes = plt.subplots(n_clusters, 1, figsize=(5, 2 * n_clusters), sharex=True)

# Iterate over each cluster and plot
for idx, cluster in enumerate(leiden_clusters):
    ax = axes[idx]
    
    # Extract data for the current cluster
    cluster_data = count_data.loc[cluster]
    
    # Create a horizontal bar plot for the current cluster
    cluster_data.plot(kind='barh', ax=ax, color=[stage_colors[col] for col in cluster_data.index])
    
    # Add counts on top of each bar
    for p in ax.patches:
        # Get the infection status from the index
        infection_status = cluster_data.index[ax.patches.index(p)]
        
        # Set text color based on infection status
        text_color = 'red' if 'infected' in infection_status else 'black'
        
        # Annotate with the determined text color
        ax.annotate(f'{int(p.get_width())}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', color=text_color)

    # Title and labels for each subplot
    ax.set_title(f'Counts of Cells for Cluster {cluster}')
    ax.set_ylabel('Count')

# Set the x-label for the shared x-axis
axes[-1].set_xlabel('Number of Cells')

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Plot top enriched pathways for each compared cluster/group
import matplotlib.pyplot as plt

# Group by 'leiden' and 'infection_status' and count occurrences
count_data = adata_subset.obs.groupby(['leiden', 'niches']).size().unstack(fill_value=0)

# Unique leiden clusters
leiden_clusters = count_data.index

# Create subplots - one for each unique leiden cluster
n_clusters = len(leiden_clusters)
fig, axes = plt.subplots(n_clusters, 1, figsize=(10, 5 * n_clusters), sharex=True)

# Iterate over each cluster and plot
for idx, cluster in enumerate(leiden_clusters):
    ax = axes[idx]
    
    # Extract data for the current cluster
    cluster_data = count_data.loc[cluster]
    
    # Create a horizontal bar plot for the current cluster
    cluster_data.plot(kind='barh', ax=ax, color=[niche_colors[col] for col in cluster_data.index])
    
    # Add counts on top of each bar
    for p in ax.patches:
        # Get the infection status from the index
        infection_status = cluster_data.index[ax.patches.index(p)]
        
        # Set text color based on infection status
        text_color = 'red' if 'infected' in infection_status else 'black'
        
        # Annotate with the determined text color
        ax.annotate(f'{int(p.get_width())}', 
                    (p.get_width(), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', color=text_color)

    # Title and labels for each subplot
    ax.set_title(f'Counts of Cells for Cluster {cluster}')
    ax.set_ylabel('Count')

# Set the x-label for the shared x-axis
axes[-1].set_xlabel('Number of Cells')

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# Re-import plotting stack for optional post-hoc visualization and checks
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

# Step 1: Calculate angiogenesis_meta
gene_list = ['COL5A2', 'PDGFA', 'ITGAV', 'POSTN']
# Make sure the genes exist in the adata_subset
if all(gene in adata_subset.var_names for gene in gene_list):
    adata_subset.obs['angiogenesis_meta'] = adata_subset[:, gene_list].X.sum(axis=1).A1  # .A1 converts sparse matrix to array
else:
    print("One or more genes are not present in the dataset.")

# Step 2: Ensure infection_status is added to adata_subset
# This is a placeholder. Replace with your actual infection status data.
# Example: adata_subset.obs['infection_status'] = np.random.choice(['Infected', 'Not Infected'], size=adata_subset.n_obs)

# Step 3: Define color mapping for infection status
infection_status_colors = {
    'infected': '#ff0000',  # Red for Infected
    'uninfected': '#D3D3D3'  # Light gray for Not Infected
}

# Set up subplot grid (3 rows, 2 columns)
fig, axes = plt.subplots(2, 3, figsize=(20, 10), dpi=150)
axes = axes.flatten()

s = 0.8

# Plot 1: Stage
sce.pl.phate(
    adata_subset,
    color='Stage',
    ax=axes[0],
    show=False, s=s,
    palette=stage_colors
)

# Plot 2: Infection Status
sce.pl.phate(
    adata_subset,
    color='infection_status',
    ax=axes[1],
    show=False, s=s,
    palette=infection_status_colors
)

# Plot 3: Niche with Tumor Proximity
sce.pl.phate(
    adata_subset,
    color='niches',
    ax=axes[2],
    show=False, s=s,
    palette=niche_colors
)

# Plot 4: Angiogenesis Meta
sce.pl.phate(
    adata_subset,
    color='angiogenesis_meta',
    ax=axes[3],
    show=False, s=s,
    palette='viridis'  # or any colormap you prefer
)

# Plot 5: Path Block Core (no palette provided)
sce.pl.phate(
    adata_subset,
    color='leiden',
    ax=axes[4],
    show=False, s=s
)

# Plot 6: Broad Cell Types
sce.pl.phate(
    adata_subset,
    color='cell_type',
    ax=axes[5],
    show=False, s=s,
    palette=sub_cell_types_color_mapping
)

# Finalize
plt.tight_layout()
plt.show()

In [ ]:
# Set output prefix used for exported trajectory artifacts and tables
output_prefix = 'LEC_VEC'

In [ ]:
# Save the trajectory subset AnnData object for downstream R/Slingshot analysis
adata_subset.write_h5ad(f'../data/KS_adata_preprocessed.h5ad')

In [ ]:
# Compute differential-expression markers across selected Leiden groups
sc.tl.rank_genes_groups(adata_subset,
                       groupby='leiden',
                       )

In [ ]:
# Compute cluster dendrogram to support rank_genes_groups heatmap ordering
sc.tl.dendrogram(adata_subset,groupby='leiden', )

In [ ]:
# Visualize ranked marker genes for selected clusters
sc.pl.rank_genes_groups_dotplot(adata_subset)

In [ ]:
# Plot PHATE embedding colored by key annotations for trajectory interpretation
sce.pl.phate(adata_subset, color='leiden')

In [ ]:
# Extract ranked marker genes for the selected Leiden group
gene_list = sc.get.rank_genes_groups_df(adata_subset, group=['0'])['names'].tolist()

In [ ]:
# Quick sanity check: print top-ranked marker genes
print(gene_list[:5])

In [ ]:
# Inspect Leiden assignments before downstream composition and marker analysis
adata_subset.obs.leiden.unique()

In [ ]:
# Initialize container for GSEA outputs across compared groups
results = {}

for group in adata_subset.obs.leiden.unique():

    gene_list = sc.get.rank_genes_groups_df(adata_subset, group=[group])['names'].tolist()
    
    while True:
        try:
            enr = gp.enrichr(gene_list=gene_list[:25], 
                            gene_sets=['GO_Biological_Process_2023', 'MSigDB_Hallmark_2020'],#, 'VART_KSHV_INFECTION_ANGIOGENIC_MARKERS_UP'], 
                            outdir=None, 
                            background=None,
                            verbose=False
                            )
            results[group] = enr.results
            break
        except Exception as e:
            print(f"Error encountered in prerank for group {group}: {e}")
            print("Retrying...")
    
    ax = gp.barplot(enr.results,
                  column="Adjusted P-value",
                  x='Gene_set', # set x axis, so you could do a multi-sample/library comparsion
                  #size=10,
                  top_term=15,
                  figsize=(4,5),
                  #title = "MSigDB_Hallmark_2020",
                  xticklabels_rot=45, # rotate xtick labels
                  #show_ring=True, # set to False to revmove outer ring
                  #marker='o',
                  color='darkred'
                 )
    
    # ax.set_xlim(0, 25)
    
    plt.title(f"Cluster: {group}")
    plt.tight_layout()
    
    # plt.savefig(f'../figures/DEG_analysis/fig3_GSEA_DE_each_niche_vs_rest_{group}.pdf', format='pdf', bbox_inches='tight')
    
    plt.show()


In [ ]:
# Drop duplicated cell IDs to keep one unique entry per cell before export
unique_indices = adata_subset.obs['cell_id'].drop_duplicates(keep='first').index
adata_subset = adata_subset[unique_indices, :]

In [ ]:
# Set cell_id as index for clean downstream table alignment and export
adata_subset.obs.set_index('cell_id', inplace=True)

In [ ]:
# Export expression, metadata, cluster, and PHATE coordinate tables for trajectory modeling
import pandas as pd

# Extract the expression matrix
expression_matrix = adata_subset.X  # If using a sparse matrix, use .A to convert to dense if needed
# If adata.X is sparse, convert it to a dense format (if it's small enough)
if hasattr(adata_subset.X, 'toarray'):
    expression_matrix = adata_subset.X.toarray()

# Create a DataFrame for the expression matrix
expression_df = pd.DataFrame(expression_matrix, index=adata_subset.obs.index, columns=adata_subset.var.index)

# Save the expression matrix as a CSV file
expression_df.to_csv(f"../data/expression_matrix_subset_{output_prefix}.csv", index=True)

In [ ]:
# Extract cell metadata
cell_metadata = adata_subset.obs
cell_metadata.to_csv(f"../data/cell_metadata_subset_{output_prefix}.csv", index=True)

# Extract gene metadata
gene_metadata = adata_subset.var
gene_metadata.to_csv(f"../data/gene_metadata_subset_{output_prefix}.csv", index=True)

# Extract gene names
gene_names = adata_subset.var.index.tolist()  # Get the gene names
gene_names_df = pd.DataFrame(gene_names, columns=["gene_name"])
gene_names_df.to_csv(f"../data/gene_names_subset_{output_prefix}.csv", header=False, index=False)

print("Data extraction complete. Files saved in 'data' directory.")

In [ ]:
# Export expression, metadata, cluster, and PHATE coordinate tables for trajectory modeling
import pandas as pd

# Extract PHATE embedding
phate_df = pd.DataFrame(
    adata_subset.obs["leiden"],
    # index=["Cell_" + str(i) for i in adata_subset.obs_names],
    index=adata_subset.obs.index,
    columns=["leiden"]  # Add more if ndim > 2
)

# Save to CSV with index
phate_df.to_csv(f"../data/leiden_cluster_ids_{output_prefix}.csv", index=True)


In [ ]:
# Export expression, metadata, cluster, and PHATE coordinate tables for trajectory modeling
import pandas as pd

# Extract PHATE embedding
phate_df = pd.DataFrame(
    adata_subset.obsm["X_phate"],
    # index=["Cell_" + str(i) for i in adata_subset.obs_names],
    index=adata_subset.obs.index,
    columns=["PHATE_1", "PHATE_2"]  # Add more if ndim > 2
)

# Save to CSV with index
phate_df.to_csv(f"../data/phate_embedding_{output_prefix}.csv", index=True)
